In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


In [6]:
df = pd.read_csv('car data.csv')
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Driven_kms     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Selling_type   301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB


In [9]:
df.describe()

,Year,Selling_Price,Present_Price,Driven_kms,Owner
count,301.000000,301.000000,301.000000,301.000000,301.000000
mean,2013.627907,4.661296,7.628472,36947.205980,0.043189
std,2.891554,5.082812,8.642584,38886.883882,0.247915
min,2003.000000,0.100000,0.320000,500.000000,0.000000
25%,2012.000000,0.900000,1.200000,15000.000000,0.000000
50%,2014.000000,3.600000,6.400000,32000.000000,0.000000
75%,2016.000000,6.000000,9.900000,48767.000000,0.000000
max,2018.000000,35.000000,92.600000,500000.000000,3.000000


In [10]:
df.isnull().sum()

Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load data
df = pd.read_csv('car data.csv')

# Check unique values for categorical data
print("Unique Car Names:", df['Car_Name'].nunique())
print("Fuel Types:", df['Fuel_Type'].unique())
print("Selling Types:", df['Selling_type'].unique())
print("Transmissions:", df['Transmission'].unique())
print("Owners:", df['Owner'].unique())
print("Years range:", df['Year'].min(), "to", df['Year'].max())

# Feature Engineering: Calculate Age of the car
# Using 2026 as current year since it's 2026
df['Car_Age'] = 2026 - df['Year']

# Drop original 'Year' and 'Car_Name' (since Car_Name has high cardinality and specific models)
# We can keep Car_Name brand if we want, but let's see some names first
print(df['Car_Name'].head(10))

# Preprocessing: One-hot encoding for categorical features
df_encoded = pd.get_dummies(df.drop(columns=['Car_Name', 'Year']), drop_first=True)

print("\nEncoded Columns:")
print(df_encoded.columns)

# Define X and y
X = df_encoded.drop(columns=['Selling_Price'])
y = df_encoded['Selling_Price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predict
y_pred = rf.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n--- Model Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

# Feature Importance
importances = rf.feature_importances_
feature_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\n--- Feature Importance ---")
print(feature_imp)

# Plot 1: Actual vs Predicted
plt.clf()
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7, color='b')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Selling Price')
plt.ylabel('Predicted Selling Price')
plt.title('Actual vs Predicted Car Prices')
plt.tight_layout()
plt.savefig('actual_vs_predicted_car_prices.png')

# Plot 2: Feature Importance
plt.clf()
sns.barplot(x='Importance', y='Feature', data=feature_imp, palette='viridis')
plt.title('Feature Importance for Car Price Prediction')
plt.tight_layout()
plt.savefig('car_feature_importance.png')

Unique Car Names: 98
Fuel Types: ['Petrol' 'Diesel' 'CNG']
Selling Types: ['Dealer' 'Individual']
Transmissions: ['Manual' 'Automatic']
Owners: [0 1 3]
Years range: 2003 to 2018
0             ritz
1              sx4
2             ciaz
3          wagon r
4            swift
5    vitara brezza
6             ciaz
7          s cross
8             ciaz
9             ciaz
Name: Car_Name, dtype: object

Encoded Columns:
Index(['Selling_Price', 'Present_Price', 'Driven_kms', 'Owner', 'Car_Age',
       'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Selling_type_Individual',
       'Transmission_Manual'],
      dtype='object')

--- Model Performance ---
Mean Absolute Error (MAE): 0.64
Mean Squared Error (MSE): 0.93
R-squared (R2) Score: 0.9595


AttributeError: 'RandomForestRegressor' object has no attribute 'feature_importances'